# Chapter 12: Autoencoders


<!-- Macro definitions for MathJax, mirroring book.tex -->
$$
\newcommand{\bm}[1]{\boldsymbol{#1}}
\newcommand{\Det}[1]{|\boldsymbol{#1}|}
\newcommand{\bigO}{\mathcal{O}}
\newcommand{\var}{\mathrm{Var}}
\newcommand{\cov}{\mathrm{Cov}}
\newcommand{\Prob}{\mathrm{Prob}}
\newcommand{\mean}[1]{\langle #1 \rangle}
$$

Every network so far has been trained against labels.  Regression needed
targets $y_i$, classification needed classes, the differential-equation solvers
of Chapter 9 needed an equation.  This chapter removes the labels
and asks the network to reproduce its own input.

That sounds vacuous -- the identity map is available for free -- and it would be,
were it not for a constraint.  An *autoencoder* factors the map through a
representation of restricted size or restricted form,

$$
\bm{x} \;\longmapsto\; \bm{z}=f_\theta(\bm{x})
        \;\longmapsto\; \hat{\bm{x}}=g_\phi(\bm{z}),\tag{12.1}
$$

and asks for the best approximation to the identity subject to that
factorisation.  The whole subject is the study of what that constraint forces
the network to discover.

The chapter has two halves.  The first is exact: when $f$ and $g$ are linear the
problem admits a complete analytic solution, and the answer is principal
component analysis.  We prove this, carefully, because the proof is short,
because it is the one place in deep learning where a global optimum is known,
and because knowing exactly what the linear case does tells us what the
nonlinear case is buying.  The second half is what happens when the linearity is
given up: deep, denoising, sparse and contractive autoencoders, the convolutional
and recurrent variants that reuse Chapters 10 and 11
unchanged, and the probabilistic reading that connects the whole construction
back to Chapter 2.

The material follows the lecture notes for weeks eight to ten of
FYS-STK3155/4155.


## Structure

An autoencoder is a network in two halves.  The *encoder*
$f_\theta:\mathbb{R}^d\to\mathbb{R}^p$ maps an input to a *code* or
*latent representation* $\bm{z}$, and the *decoder*
$g_\phi:\mathbb{R}^p\to\mathbb{R}^d$ maps the code back.  The two are trained
together against the reconstruction error, so that for a data set
$\{\bm{x}_n\}_{n=1}^{N}$ we minimise

$$
\boxed{\;
  \mathcal{L}(\theta,\phi) = \frac{1}{N}\sum_{n=1}^{N}
    \left\|\bm{x}_n - g_\phi\!\left(f_\theta(\bm{x}_n)\right)\right\|_2^{2}.\;}\tag{12.2}
$$

No labels appear.  The target of example $n$ is example $n$, which is why the
method is called *self-supervised*: the supervision is manufactured from
the data.

The layer that produces $\bm{z}$ is the *bottleneck*, and its width $p$ is
the single most important architectural choice.  Three regimes are worth naming.

- *Undercomplete*, $p<d$.  The code cannot hold the input, so the
   network must decide what to discard.  This is the case in which reconstruction
   forces compression, and it is the case the first half of this chapter analyses
   exactly.
- *Complete*, $p=d$.  Nothing is forced.
- *Overcomplete*, $p\ge d$.  The identity is available:
   $f=g=\mathrm{id}$ gives zero error and learns nothing.  Something other than
   the bottleneck must supply the constraint, and Section *Regularised autoencoders*
   is about what.

In practice the two halves are usually *mirrored*: a decoder whose layer
widths are those of the encoder in reverse.  This is a convention rather than a
requirement, but it is a sensible one, and it makes the parameter counts of the
two halves match.

**Activations and the cost.** 
The hidden layers use whatever Chapter 8 recommends.  The
*output* layer is the one that needs thought, and it should be chosen to
match the range of the data.

If the inputs are unbounded real numbers, the output activation is the identity
and the cost is Eq. (12.2).  If they have been scaled to $[0,1]$ --
pixel intensities, most commonly -- a logistic output is natural, and the
binary cross-entropy

$$
\mathcal{L}_{\mathrm{BCE}} = -\frac{1}{N}\sum_{n=1}^{N}\sum_{j=1}^{d}
    \left[x_{nj}\log\hat{x}_{nj}
        + (1-x_{nj})\log(1-\hat{x}_{nj})\right]\tag{12.3}
$$

is then often preferred to the squared error, because it is the negative
log-likelihood of a Bernoulli model and it penalises confident mistakes near
the ends of the range much more heavily.  A ReLU output is appropriate when the
data are non-negative and unbounded, and a linear output is wrong for bounded
data because it can predict outside the range.


## Reconstruction, projection and representation

Before the analysis it is worth being precise about what is being asked.  The
optimisation is over function classes: let $\mathcal{F}$ be a class of encoders
and $\mathcal{G}$ a class of decoders, and solve

$$
\min_{f\in\mathcal{F},\,g\in\mathcal{G}}
    \sum_{n=1}^{N}\left\|\bm{x}_n-g(f(\bm{x}_n))\right\|_2^{2}.\tag{12.4}
$$

If $\mathcal{F}$ and $\mathcal{G}$ are linear this is a constrained quadratic
problem with a closed-form solution.  If they are neural networks it is
non-convex and highly expressive, and we are back in the territory of
Chapter 4.

A theme that will recur is the decomposition

$$
\left\|\bm{X}-\hat{\bm{X}}\right\|_F^{2}
  = \underbrace{\text{total variance}}_{\text{fixed by the data}}
  - \underbrace{\text{explained variance}}_{\text{what the code captures}},\tag{12.5}
$$

which says that minimising reconstruction error and maximising captured
variance are the same problem seen from two sides.  For linear autoencoders this
is exact and we prove it below.

```{admonition} Reconstruction is not representation
:class: tip
Equation (12.2)
scores the network on $\hat{\bm{x}}$, not on $\bm{z}$.  Nothing in it asks the
code to be interpretable, smooth, disentangled or ordered, and
Section *Beyond PCA* shows an example in which the reconstruction is
excellent and the code is nevertheless a poor coordinate for the data.  A low
reconstruction error is evidence that the information survived the bottleneck.
It is not evidence that it survived in a useful form.
```


## The linear autoencoder

Take the encoder and decoder to be linear and without biases,

$$
\bm{z} = \bm{W}_e^{\mathsf{T}}\bm{x},
  \qquad
  \hat{\bm{x}} = \bm{W}_d^{\mathsf{T}}\bm{z},
  \qquad
  \bm{W}_e\in\mathbb{R}^{d\times p},\;
  \bm{W}_d\in\mathbb{R}^{p\times d},\tag{12.6}
$$

so that the reconstruction map is $\hat{\bm{x}}=\bm{A}\bm{x}$ with
$\bm{A}=\bm{W}_d^{\mathsf{T}}\bm{W}_e^{\mathsf{T}}$.  Since $\bm{A}$ factors
through $\mathbb{R}^{p}$,

$$
\operatorname{rank}(\bm{A}) \le p .\tag{12.7}
$$

Dropping the biases costs nothing provided the data are centred, which we assume
throughout: replace $\bm{x}_n$ by $\bm{x}_n-\bar{\bm{x}}$.  With the data
arranged as the rows of $\bm{X}\in\mathbb{R}^{N\times d}$, the training problem
is

$$
\min_{\bm{W}_e,\bm{W}_d}
    \left\|\bm{X}-\bm{X}\bm{A}\right\|_F^{2}
  \;=\;
  \min_{\operatorname{rank}(\bm{A})\le p}
    \left\|\bm{X}-\bm{X}\bm{A}\right\|_F^{2}.\tag{12.8}
$$

The equality is the first substantive step: *any* matrix of rank at most
$p$ can be written as $\bm{W}_d^{\mathsf{T}}\bm{W}_e^{\mathsf{T}}$ by taking a
rank factorisation, so optimising over the two factors is the same as optimising
over their product subject to the rank bound.  Linear autoencoder training is a
rank-constrained approximation problem.

Write $\bm{S}=\bm{X}^{\mathsf{T}}\bm{X}/N$ for the empirical covariance, and let
its eigendecomposition be

$$
\bm{S} = \bm{U}\bm{\Lambda}\bm{U}^{\mathsf{T}},
  \qquad
  \lambda_1\ge\lambda_2\ge\dots\ge\lambda_d\ge0,\tag{12.9}
$$

with $\bm{U}$ orthogonal and $\bm{U}_p=[\bm{u}_1,\dots,\bm{u}_p]$ its first $p$
columns.  Recall from Chapter 1 that $\bm{S}$ is symmetric
positive semi-definite, so such a decomposition exists with real non-negative
eigenvalues.

### Reduction to an orthogonal projector

```{admonition} Lemma (The optimum may be taken to be an orthogonal projector)
:class: important
Let $\bm{A}$ have rank at most $p$ and let $\bm{P}$ be the orthogonal projector
onto the row space of $\bm{A}$ (equivalently, onto the image of
$\bm{A}^{\mathsf{T}}$).  Then $\operatorname{rank}(\bm{P})\le p$ and

$$
\left\|\bm{X}-\bm{X}\bm{P}\right\|_F^{2}
  \;\le\;
  \left\|\bm{X}-\bm{X}\bm{A}\right\|_F^{2}.\tag{12.10}
$$
```

```{admonition} Proof
:class: note
Let $\mathcal{M}$ be the row space of $\bm{A}$, of dimension at most $p$.  For
each $n$, the row $\bm{x}_n^{\mathsf{T}}\bm{A}$ lies in $\mathcal{M}$, and
$\bm{x}_n^{\mathsf{T}}\bm{P}$ is by definition the point of $\mathcal{M}$
closest to $\bm{x}_n$ in the Euclidean norm.  Hence

$$
\left\|\bm{x}_n-\bm{P}\bm{x}_n\right\|_2^{2}
  \le \left\|\bm{x}_n-\bm{A}^{\mathsf{T}}\bm{x}_n\right\|_2^{2}
  \qquad\text{for every }n,
$$

and summing over $n$ gives Eq. (12.10), the Frobenius norm being
the sum over rows of the squared Euclidean norms.
```

Lemma lem:12-projector is what makes the whole analysis possible.  It
replaces an optimisation over all rank-$p$ matrices, a set with $p(2d-p)$
parameters and no useful structure, by an optimisation over rank-$p$ orthogonal
projectors, which are parameterised by a Grassmannian -- that is, by a choice of
subspace and nothing else.  Every orthogonal projector of rank $p$ has the form

$$
\bm{P}=\bm{B}\bm{B}^{\mathsf{T}},
  \qquad
  \bm{B}\in\mathbb{R}^{d\times p},
  \quad \bm{B}^{\mathsf{T}}\bm{B}=\bm{I}_p,\tag{12.11}
$$

with $\bm{B}$ an orthonormal basis of the subspace.

### Reduction to a trace maximisation

```{admonition} Lemma (Error equals variance not captured)
:class: important
With $\bm{P}=\bm{B}\bm{B}^{\mathsf{T}}$ as in Eq. (12.11),

$$
\left\|\bm{X}-\bm{X}\bm{P}\right\|_F^{2}
  = N\left[\operatorname{Tr}(\bm{S})
    - \operatorname{Tr}\!\left(\bm{B}^{\mathsf{T}}\bm{S}\bm{B}\right)\right].\tag{12.12}
$$

Minimising the reconstruction error is therefore equivalent to maximising
$\operatorname{Tr}(\bm{B}^{\mathsf{T}}\bm{S}\bm{B})$ subject to
$\bm{B}^{\mathsf{T}}\bm{B}=\bm{I}_p$.
```

```{admonition} Proof
:class: note
Expand the norm as a trace and use $\bm{P}^{\mathsf{T}}=\bm{P}$ and
$\bm{P}^{2}=\bm{P}$:

$$
\left\|\bm{X}-\bm{X}\bm{P}\right\|_F^{2}
  = \operatorname{Tr}\!\left[(\bm{I}-\bm{P})\bm{X}^{\mathsf{T}}
      \bm{X}(\bm{I}-\bm{P})\right]
  = N\operatorname{Tr}\!\left[(\bm{I}-\bm{P})\bm{S}(\bm{I}-\bm{P})\right],
$$

using $\bm{X}^{\mathsf{T}}\bm{X}=N\bm{S}$.  Idempotence and cyclicity of the
trace give
$\operatorname{Tr}[(\bm{I}-\bm{P})\bm{S}(\bm{I}-\bm{P})]
=\operatorname{Tr}[\bm{S}(\bm{I}-\bm{P})]
=\operatorname{Tr}(\bm{S})-\operatorname{Tr}(\bm{S}\bm{P})$,
and finally
$\operatorname{Tr}(\bm{S}\bm{B}\bm{B}^{\mathsf{T}})
=\operatorname{Tr}(\bm{B}^{\mathsf{T}}\bm{S}\bm{B})$.
```

Equation (12.12) is Eq. (12.5) made
precise: $\operatorname{Tr}(\bm{S})$ is the total variance, fixed by the data,
and $\operatorname{Tr}(\bm{B}^{\mathsf{T}}\bm{S}\bm{B})$ is the variance
captured by the subspace.

### The variational characterisation

It remains to maximise $\operatorname{Tr}(\bm{B}^{\mathsf{T}}\bm{S}\bm{B})$.
We do the case $p=1$ by a Lagrange multiplier, which is where the eigenvalue
problem appears, and then state the general case.

```{admonition} Proposition (The first direction)
:class: important
The maximiser of $\bm{w}^{\mathsf{T}}\bm{S}\bm{w}$ subject to
$\|\bm{w}\|_2=1$ is the eigenvector $\bm{u}_1$ of $\bm{S}$ belonging to the
largest eigenvalue $\lambda_1$, and the maximum equals $\lambda_1$.
```

```{admonition} Proof
:class: note
Without the constraint the objective is unbounded, since scaling $\bm{w}$ scales
it quadratically; the constraint is therefore essential.  Impose it with a
multiplier and maximise

$$
J(\bm{w}) = \bm{w}^{\mathsf{T}}\bm{S}\bm{w}
            + \lambda\left(1-\bm{w}^{\mathsf{T}}\bm{w}\right).
$$

By the matrix-calculus identities of Section *Derivatives with respect to vectors and matrices*,

$$
\frac{\partial J}{\partial\bm{w}} = 2\bm{S}\bm{w}-2\lambda\bm{w} = \bm{0}
  \quad\Longleftrightarrow\quad
  \bm{S}\bm{w}=\lambda\bm{w},
$$

so every stationary point is an eigenvector of $\bm{S}$.  Left-multiplying by
$\bm{w}^{\mathsf{T}}$ and using $\|\bm{w}\|=1$ gives
$\bm{w}^{\mathsf{T}}\bm{S}\bm{w}=\lambda$: the value of the objective at a
stationary point *is* the corresponding eigenvalue.  The maximum is
therefore attained at the largest one.
```

The direction that maximises the variance of the projected data is an
eigenvector of the covariance matrix, and the variance it captures is the
eigenvalue.  That single sentence is the content of principal component
analysis.

```{admonition} Theorem (Ky Fan)
:class: important
Let $\bm{S}$ be symmetric with eigenvalues
$\lambda_1\ge\dots\ge\lambda_d$ and eigenvectors $\bm{u}_1,\dots,\bm{u}_d$.
Then

$$
\max_{\bm{B}^{\mathsf{T}}\bm{B}=\bm{I}_p}
    \operatorname{Tr}\!\left(\bm{B}^{\mathsf{T}}\bm{S}\bm{B}\right)
  = \sum_{i=1}^{p}\lambda_i,\tag{12.13}
$$

attained at $\bm{B}=\bm{U}_p$, and at $\bm{B}=\bm{U}_p\bm{Q}$ for any orthogonal
$\bm{Q}\in\mathbb{R}^{p\times p}$.
```

```{admonition} Proof
:class: note
Write $\bm{C}=\bm{U}^{\mathsf{T}}\bm{B}$, which also satisfies
$\bm{C}^{\mathsf{T}}\bm{C}=\bm{I}_p$ since $\bm{U}$ is orthogonal.  Then

$$
\operatorname{Tr}\!\left(\bm{B}^{\mathsf{T}}\bm{S}\bm{B}\right)
  = \operatorname{Tr}\!\left(\bm{C}^{\mathsf{T}}\bm{\Lambda}\bm{C}\right)
  = \sum_{i=1}^{d}\lambda_i\,m_i,
  \qquad
  m_i := \sum_{j=1}^{p}C_{ij}^{2}.
$$

The coefficients $m_i$ satisfy two constraints.  Each row of $\bm{C}$ has norm
at most one, because $\bm{C}$ has orthonormal columns and may be completed to an
orthogonal matrix, so $0\le m_i\le1$; and
$\sum_i m_i=\|\bm{C}\|_F^{2}=\operatorname{Tr}(\bm{C}^{\mathsf{T}}\bm{C})=p$.
Maximising the linear functional $\sum_i\lambda_i m_i$ over the polytope
$\{0\le m_i\le1,\ \sum_i m_i=p\}$ puts all the available mass on the largest
$\lambda_i$, giving $m_1=\dots=m_p=1$ and the rest zero, hence the value
$\sum_{i\le p}\lambda_i$.  This is attained by $\bm{C}=[\bm{I}_p;\bm{0}]$, that
is $\bm{B}=\bm{U}_p$, and the value is unchanged by $\bm{B}\to\bm{B}\bm{Q}$ with
$\bm{Q}$ orthogonal since $\operatorname{Tr}(\bm{Q}^{\mathsf{T}}
\bm{B}^{\mathsf{T}}\bm{S}\bm{B}\bm{Q})
=\operatorname{Tr}(\bm{B}^{\mathsf{T}}\bm{S}\bm{B})$.
```

### The theorem

Assembling Lemmas lem:12-projector and lem:12-trace with
Theorem thm:12-kyfan gives the result.

```{admonition} Theorem (Linear autoencoders perform PCA)
:class: important
Let $\bm{X}\in\mathbb{R}^{N\times d}$ be centred with empirical covariance
$\bm{S}=\bm{X}^{\mathsf{T}}\bm{X}/N$, eigenvalues
$\lambda_1\ge\dots\ge\lambda_d\ge0$ and eigenvectors $\bm{U}$.  Consider the
linear autoencoder (12.6) with bottleneck width $p\le d$ trained on
the reconstruction error (12.2).  Then:

1. the minimum of the reconstruction error is

   $$
   \min_{\bm{W}_e,\bm{W}_d}\frac{1}{N}
   \left\|\bm{X}-\bm{X}\bm{W}_e\bm{W}_d\right\|_F^{2}
   = \sum_{i=p+1}^{d}\lambda_i,\tag{12.14}
   $$

   the sum of the discarded eigenvalues;
2. at any global minimiser the product satisfies
   $\bm{W}_e\bm{W}_d=\bm{U}_p\bm{U}_p^{\mathsf{T}}$, the orthogonal projector
   onto the leading eigenspace, and the reconstruction is therefore identical to
   the rank-$p$ PCA reconstruction $\hat{\bm{x}}
   =\bm{U}_p\bm{U}_p^{\mathsf{T}}\bm{x}$;
3. the column space of $\bm{W}_e$ equals
   $\operatorname{span}\{\bm{u}_1,\dots,\bm{u}_p\}$, assuming
   $\lambda_p>\lambda_{p+1}$.
```

```{admonition} Proof
:class: note
By Eq. (12.8) the problem is over matrices of rank at most
$p$; by Lemma lem:12-projector the infimum is attained on orthogonal
projectors, which by Eq. (12.11) are parameterised by
$\bm{B}$ with orthonormal columns; by Lemma lem:12-trace the objective
becomes $N[\operatorname{Tr}(\bm{S})
-\operatorname{Tr}(\bm{B}^{\mathsf{T}}\bm{S}\bm{B})]$; and by
Theorem thm:12-kyfan the second trace is maximised at $\sum_{i\le
p}\lambda_i$ with $\bm{B}=\bm{U}_p\bm{Q}$.  Since
$\operatorname{Tr}(\bm{S})=\sum_i\lambda_i$, the minimum value is
$N\sum_{i>p}\lambda_i$, which after division by $N$ is
Eq. (12.14).  The optimal projector is
$\bm{B}\bm{B}^{\mathsf{T}}=\bm{U}_p\bm{Q}\bm{Q}^{\mathsf{T}}\bm{U}_p^{\mathsf{T}}
=\bm{U}_p\bm{U}_p^{\mathsf{T}}$, independent of $\bm{Q}$, which gives (ii);
the gap condition $\lambda_p>\lambda_{p+1}$ makes the maximising subspace unique
and gives (iii).
```

The result deserves a moment's reflection.  A neural network trained by gradient
descent on a non-convex-looking objective is nevertheless guaranteed to land on
a classical, spectrally characterised answer.  It is essentially the only
architecture in this book for which we can say that.

```{admonition} Proposition (What the autoencoder does *not* recover)
:class: important
If $(\bm{W}_e,\bm{W}_d)$ is a global minimiser then so is
$(\bm{W}_e\bm{M},\bm{M}^{-1}\bm{W}_d)$ for every invertible
$\bm{M}\in\mathbb{R}^{p\times p}$.  Consequently the individual weights are
determined only up to $GL(p)$: the autoencoder recovers the principal
*subspace*, not the principal *directions*, and its codes are not in
general uncorrelated, ordered by variance, or of unit norm.
```

```{admonition} Proof
:class: note
$(\bm{W}_e\bm{M})(\bm{M}^{-1}\bm{W}_d)=\bm{W}_e\bm{W}_d$, so the reconstruction
and hence the cost are unchanged.
```

This is the practical difference between running PCA and training a linear
autoencoder, and it is easy to overlook.  PCA delivers an orthonormal basis
ordered by variance, so that truncating to $q<p$ components is meaningful and
the components are uncorrelated by construction.  A linear autoencoder delivers
some basis of the same subspace, in no particular order, generally not
orthogonal.  If the directions matter, use PCA; if only the subspace matters,
either will do, and PCA is faster.

```{admonition} Relation to Eckart-Young and the SVD
:class: tip
Theorem thm:12-aepca is a close relative of the Eckart-Young theorem of
Section *The singular value decomposition*, which states that the best rank-$p$ approximation to
$\bm{X}$ in the Frobenius norm is obtained by truncating its singular value
decomposition, with error $\sum_{i>p}\sigma_i^{2}$.  The two are connected by
$\sigma_i^{2}=N\lambda_i$: the singular values of the centred data matrix are
the square roots of $N$ times the eigenvalues of the covariance.  The difference
is that Eckart-Young approximates $\bm{X}$ by *any* low-rank matrix, while
the autoencoder is constrained to approximate each $\bm{x}_n$ by a fixed linear
map applied to $\bm{x}_n$ itself.  That the two give the same answer is a
consequence of Lemma lem:12-projector.  In practice one computes PCA by
the SVD of $\bm{X}$ rather than by eigendecomposing $\bm{S}$, for the
conditioning reason given in Section *The singular value decomposition*: forming
$\bm{X}^{\mathsf{T}}\bm{X}$ squares the condition number.
```


## Verifying the theorem

The chapter's code is the network of Chapter 8 with the target set
equal to the input.  Nothing else changes, which is the point.


In [ ]:
def ae_forward(P, X, acts):
    """Returns the reconstruction and the cache; X has shape (N, d)."""
    A = [X]; Z = []
    a = X
    for l, (W, b) in enumerate(P):
        z = a @ W + b
        Z.append(z)
        a = ACT[acts[l]][0](z)
        A.append(a)
    return a, (A, Z)


def ae_backward(P, cache, Xhat, X, acts):
    """Backpropagation with the target equal to the input, Eq. (12.cost)."""
    A, Z = cache
    n = X.shape[0]
    g = [[np.zeros_like(W), np.zeros_like(b)] for W, b in P]
    delta = (Xhat - X) / n * ACT[acts[-1]][1](Z[-1])
    for l in reversed(range(len(P))):
        g[l][0] = A[l].T @ delta
        g[l][1] = delta.sum(axis=0)
        if l > 0:
            delta = (delta @ P[l][0].T) * ACT[acts[l - 1]][1](Z[l - 1])
    return g


Checking every parameter against central differences, for three architectures:


```
  [6, 3, 6]          ['identity', 'identity']
      max relative error 8.71e-09   max absolute error 6.93e-10
      (9 entries had gradient 0 and were checked absolutely)
  [6, 4, 2, 4, 6]    ['tanh', 'tanh', 'tanh', 'sigmoid']
      max relative error 1.91e-06   max absolute error 6.19e-10
  [6, 3, 6]          ['relu', 'identity']
      max relative error 2.12e-08   max absolute error 7.35e-10
```


```{admonition} A relative check is meaningless on a gradient that is zero
:class: tip
The first line above reports nine parameters excluded from the relative
comparison.  They are the biases of a linear autoencoder on centred data, whose
gradients are *exactly* zero -- the analytic value came out as
$4.9\times10^{-17}$ and the central difference as $-4.4\times10^{-10}$, pure
subtraction noise.  Their ratio is $\bigO(1)$ and means nothing.  This is the
same lesson as the ReLU kink of Section *Verifying the implementation*, in a different
disguise: a gradient checker needs an absolute test as well as a relative one,
and needs to know which to apply where.
```

Now the theorem.  We generate $600$ points in $\mathbb{R}^{8}$ lying near a
three-dimensional subspace, plus isotropic noise, and train a
$[8,3,8]$ linear autoencoder by Adam.  Against the predictions of
Theorem thm:12-aepca and Proposition prop:12-nonunique:


```
AE  reconstruction error : 0.308313
PCA reconstruction error : 0.306324
eigenvalue tail          : 0.306324
AE/PCA error ratio       : 1.006493
principal angles (deg)   : [0.0684 0.1851 0.318 ]
||W_e W_d - Pi||_F       : 9.895e-03
singular values of W_e W_d: [1.0033 1.0007 0.9989 1.166e-16 6.462e-17
                             5.420e-17 3.645e-17 1.840e-18]
||W_e - U_p||_F          : 1.8737  (NOT small)
dist(W_e, span U_p)      : 2.466e-03  (small: same subspace)
```


Every line is a prediction confirmed.  The PCA reconstruction error and the
eigenvalue tail agree to all six digits printed, which is
Eq. (12.14).  The autoencoder gets to within $0.65\%$ of that floor
and does not cross it.  The singular values of $\bm{W}_e\bm{W}_d$ are three ones
and five zeros *to machine precision*, so the trained product really is an
orthogonal projector of rank three, as Lemma lem:12-projector said it
would be.  The principal angles between the learned subspace and the leading
eigenspace are a fifth of a degree.

And the last two lines are Proposition prop:12-nonunique in numbers.  The
distance from $\bm{W}_e$ to $\bm{U}_p$ is $1.87$, which is not small -- the
weights are nowhere near the eigenvectors -- while the distance from $\bm{W}_e$
to the *span* of $\bm{U}_p$ is $2.5\times10^{-3}$.  Same subspace,
different basis.  A reader who trained a linear autoencoder expecting to read
principal directions off the encoder weights would get nonsense, and the
reconstruction error would give no hint of it.

![a The eigenvalue spectrum of bmS on a logarithmic scale the red bars a](../BookML/BookFigures/chapter12_autoencoders/ae_pca.png)

*(a) The eigenvalue spectrum of $\bm{S}$ on a logarithmic scale; the red bars are the discarded eigenvalues whose sum is the error floor of Eq. (12.14).  (b) The autoencoder's reconstruction error minus that floor during training: it approaches zero from above and never crosses, as Theorem thm:12-aepca guarantees.  (c) Singular values of the trained product $\bm{W}_e\bm{W}_d$: exactly $p=3$ of them equal one and the remaining $d-p=5$ are at the level of rounding, confirming that the learned map is an orthogonal projector of rank $p$.*


## Beyond PCA

PCA finds the best linear subspace.  Many data sets do not lie near one: images
of a rotating object, molecular conformations, solutions of a nonlinear partial
differential equation parameterised by a boundary condition, speech.  These lie
near a curved manifold, and no linear subspace of the manifold's dimension will
follow it.

Replacing the linear maps of Eq. (12.6) by the deep networks of
Chapter 8,

$$
\bm{z}=f_\theta(\bm{x})
   = \sigma_L\!\left(\bm{W}_L\sigma_{L-1}\!\left(\cdots
      \sigma_1(\bm{W}_1\bm{x}+\bm{b}_1)\cdots\right)+\bm{b}_L\right),
  \qquad
  \hat{\bm{x}}=g_\phi(\bm{z}),\tag{12.15}
$$

buys the ability to bend.  It also costs the entire analysis of
Section *The linear autoencoder*: the problem becomes non-convex, no closed form is
available, and we are back to gradient descent with no guarantee of a global
optimum.

**What replaces the projector.** 
If $f$ and $g$ are differentiable then near a point $\bm{x}_0$, with
$\bm{z}_0=f(\bm{x}_0)$,

$$
g(f(\bm{x})) \approx g(f(\bm{x}_0))
    + \bm{J}_g(\bm{z}_0)\bm{J}_f(\bm{x}_0)\left(\bm{x}-\bm{x}_0\right),\tag{12.16}
$$

so the product of Jacobians $\bm{J}_g\bm{J}_f$ is the local reconstruction
operator.  It plays the role that $\bm{W}_e\bm{W}_d$ played in the linear case,
with the crucial difference that it now *varies from point to point*.  A
nonlinear autoencoder is, to first order, a projector onto the tangent space of
the data manifold, chosen afresh at every location.

**Measuring the difference.** 
We put $800$ training and $400$ test points on a one-dimensional curve --- a
helix --- embedded in $\mathbb{R}^{3}$ with small isotropic noise, and compare
PCA against a $[3,16,p,16,3]$ autoencoder with $\tanh$ hidden layers.

| \noalign{}
$p$ | Method | train error | test error |
|---|---|---|---|
| \noalign{}\noalign{}
$1$ | PCA | $0.7855$ | $0.7914$ |
| $1$ | nonlinear autoencoder | $0.1511$ | $\mathbf{0.2012}$ |
| \noalign{}\noalign{}
$2$ | PCA | $0.3208$ | $0.3316$ |
| $2$ | nonlinear autoencoder | $0.0020$ | $\mathbf{0.0062}$ |
| \noalign{} |

*A one-dimensional curved manifold in $\mathbb{R}^{3}$.  Errors are
$\|\bm{x}-\hat{\bm{x}}\|^{2}$ averaged over points, and the autoencoder figures
are means over three seeds.  The total variance per point is $1.287$.*

At $p=1$ the autoencoder is four times more accurate than PCA; at $p=2$ it is
fifty times more accurate.  The sharpest way to put it is to compare across
rows: the *one*-dimensional nonlinear code, at $0.2012$, beats the
*two*-dimensional linear one, at $0.3316$.  Curvature is worth more than a
dimension here, and Theorem thm:12-aepca tells us exactly why PCA cannot
do better -- its error is fixed by the eigenvalue tail, and no amount of
training changes it.

![a The data, coloured by position along the curve, with the one-dimensi](../BookML/BookFigures/chapter12_autoencoders/ae_nonlinear.png)

*(a) The data, coloured by position along the curve, with the one-dimensional PCA reconstruction in red: a straight line through a helix. (b) The same data with the one-dimensional nonlinear autoencoder reconstruction, which follows the curve.  (c) The learned code $z$ against the true parameter $t$.  The relation is monotone in stretches but saturates at $\pm1$ and jumps: the reconstruction is good and the coordinate is not.*

Panel (c) of Figure fig:aenonlinear is the honest part, and it is the
notebox of Section *Reconstruction, projection and representation* made visible.  The autoencoder
reconstructs the helix well, but the code it uses to do so is not a faithful
coordinate along the curve: it saturates against the limits of the $\tanh$ and
jumps between branches.  Nothing in Eq. (12.2) asked for a good
coordinate, so nothing delivered one.  If the latent variable is to be
interpreted -- as a reaction coordinate, an order parameter, a physical
parameter of the system -- then reconstruction error alone is the wrong
objective, and one of the regularisers of the next section, or the probabilistic
treatment of Section *The probabilistic view: PPCA*, must be brought in.


## Regularised autoencoders

The bottleneck is one way to prevent an autoencoder from learning the identity.
It is not the only way, and an *overcomplete* autoencoder with $p\ge d$
needs another, since $g\circ f=\mathrm{id}$ is available and costs nothing.  All
three constructions below add a penalty to Eq. (12.2) and can
replace the bottleneck entirely.

**Denoising autoencoders.** 

Corrupt the input and ask for the clean version back.  With
$\tilde{\bm{x}}=\bm{x}+\bm{\varepsilon}$,

$$
\min_{\theta,\phi}\frac{1}{N}\sum_{n=1}^{N}
    \left\|\bm{x}_n-g_\phi\!\left(f_\theta(\tilde{\bm{x}}_n)\right)\right\|_2^{2}.\tag{12.17}
$$

Copying the input is now actively harmful, because the input is wrong.  The
network must instead learn where the data *are*, so as to move a corrupted
point back onto them.  There is a striking statistical reading of this: for
small Gaussian corruption the optimal reconstruction satisfies

$$
r(\bm{x}) - \bm{x} \;\approx\; \sigma^{2}\nabla_{\bm{x}}\log p(\bm{x}),\tag{12.18}
$$

so the displacement the network applies is proportional to the *score* of
the data density.  A denoising autoencoder is, without being told so, an
estimator of $\nabla\log p$ -- which is exactly the object that score-based and
diffusion generative models are built on.

**Sparse autoencoders.** 

Penalise the code rather than the reconstruction:

$$
\mathcal{L} = \frac{1}{N}\sum_{n}\left\|\bm{x}_n-\hat{\bm{x}}_n\right\|_2^{2}
    + \lambda\sum_{n}\left\|\bm{z}_n\right\|_1 .\tag{12.19}
$$

The $\ell_1$ penalty is the one that produced exact zeros in
Section *The Lasso*, and it does the same here: each input activates only a
few latent units, and those units tend to become recognisable feature detectors.
The construction is close kin to dictionary learning and sparse coding.

**Contractive autoencoders.** 

Penalise the sensitivity of the encoder:

$$
\mathcal{L} = \frac{1}{N}\sum_{n}\left\|\bm{x}_n-\hat{\bm{x}}_n\right\|_2^{2}
    + \lambda\sum_{n}\left\|\bm{J}_f(\bm{x}_n)\right\|_F^{2}.\tag{12.20}
$$

The two terms pull in opposite directions and the geometry of the compromise is
the point.  Reconstruction requires the encoder to be sensitive to displacements
*along* the data manifold, since those change which point is being
encoded.  The penalty asks it to be insensitive in *all* directions.  The
optimum keeps sensitivity where it is needed and suppresses it elsewhere, so
that $\bm{J}_f$ ends up large on the tangent space of the manifold and small on
the normal directions.  The encoder learns the manifold's tangent structure.

```{admonition} These are all the same idea
:class: tip
A bottleneck restricts the code by
dimension; sparsity restricts it by how many coordinates may be non-zero;
contraction restricts it by how fast it may vary; denoising restricts it by
requiring robustness.  In each case the network is prevented from being the
identity and must spend its capacity on structure instead.  This is the
bias-variance trade-off of Section *The bias-variance tradeoff* appearing once more,
with the regulariser choosing which functions are cheap.
```


## The probabilistic view: PPCA

Everything so far has been deterministic: a code $\bm{z}=f(\bm{x})$ and a
reconstruction $\hat{\bm{x}}=g(\bm{z})$.  A probabilistic autoencoder replaces
these by a latent-variable model
$p_\theta(\bm{x},\bm{z})=p_\theta(\bm{x}\mid\bm{z})p(\bm{z})$, turning
representation learning into statistical inference of the kind
Chapter 2 set up.

The linear case again has a complete answer.  *Probabilistic PCA* assumes

$$
\bm{z}\sim\mathcal{N}(\bm{0},\bm{I}_p),
  \qquad
  \bm{x} = \bm{W}\bm{z}+\bm{\mu}+\bm{\epsilon},
  \qquad
  \bm{\epsilon}\sim\mathcal{N}(\bm{0},\sigma^{2}\bm{I}_d),\tag{12.21}
$$

so that $p(\bm{x}\mid\bm{z})=\mathcal{N}(\bm{W}\bm{z}+\bm{\mu},
\sigma^{2}\bm{I}_d)$.  Integrating out the latent variable, which is a Gaussian
marginalisation and therefore exact,

$$
\bm{x}\sim\mathcal{N}\!\left(\bm{\mu},\;
    \bm{W}\bm{W}^{\mathsf{T}}+\sigma^{2}\bm{I}_d\right).\tag{12.22}
$$

The model therefore describes the covariance as a low-rank term plus isotropic
noise, $\bm{\Sigma}=\bm{W}\bm{W}^{\mathsf{T}}+\sigma^{2}\bm{I}$, which is the
probabilistic statement of "low-dimensional structure plus measurement error".

Maximum-likelihood estimation in this model, worked out by Tipping and
Bishop [tipping1999], gives a loading matrix whose column space is exactly
the principal subspace, and $\hat{\sigma}^{2}$ equal to the average of the
discarded eigenvalues,

$$
\hat{\sigma}^{2} = \frac{1}{d-p}\sum_{i=p+1}^{d}\lambda_i .\tag{12.23}
$$

Compare Eq. (12.14): the same eigenvalue tail that bounded the
autoencoder's reconstruction error reappears as the estimated noise variance.
As $\sigma^{2}\to0$ the model degenerates to the deterministic projection, and
PPCA becomes ordinary PCA.  Once again $\bm{W}$ is determined only up to an
orthogonal factor, for the reason given in
Proposition prop:12-nonunique.

The gain is what a probabilistic model always gives: a likelihood, hence a
principled way to choose $p$; a generative model one can sample from; and a
proper treatment of missing data.  Replacing the linear map in
Eq. (12.21) by a neural network, and the exact marginalisation by a
variational bound, gives the variational autoencoder, which belongs to a later
discussion of generative models.


## Implementations

### Our own

The complete autoencoder is the code of Section *Verifying the theorem* with a list
of layer widths and activations.  A deep autoencoder on the helix of
Table 12.1 is four lines:


In [ ]:
acts = ["tanh", "tanh", "tanh", "identity"]
P = init_ae([3, 16, 1, 16, 3], acts, np.random.default_rng(0))
P, hist = train_ae(P, X, acts, n_epoch=600, batch=32, eta=5e-3, Xval=Xte)
Z = encode(P, X, acts, layer=2)          # the code: run the first two layers


Setting \verb!acts = ["identity", "identity"]! and
\verb!sizes = [d, p, d]! recovers the linear autoencoder of
Theorem thm:12-aepca, and \verb!pca(X, p)! in the same module returns the
projector, the loadings and the eigenvalues for comparison.  The two are worth
running side by side once.

```{admonition} Centre the data
:class: tip
Theorem thm:12-aepca assumes it, and PCA
without it finds the direction of the mean rather than the direction of greatest
variance.  Scikit-learn's \verb!PCA! centres internally; our
\verb!pca! does so too; a hand-written implementation or a neural network
without biases does not.  If the features have incomparable units, standardise
as well as centre, or the covariance will be dominated by whichever feature
happens to be measured in small units.
```

### PyTorch and TensorFlow

The first thing worth doing in a library is the experiment of
Section *Verifying the theorem*: a linear autoencoder, which should reproduce PCA.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

# three-dimensional data lying near a plane
np.random.seed(4)
def generate_3d_data(m, w1=0.1, w2=0.3, noise=0.1):
    angles = np.random.rand(m) * 3 * np.pi / 2 - 0.5
    data = np.empty((m, 3))
    data[:, 0] = np.cos(angles) + np.sin(angles)/2 + noise*np.random.randn(m)/2
    data[:, 1] = np.sin(angles) * 0.7 + noise * np.random.randn(m) / 2
    data[:, 2] = data[:, 0]*w1 + data[:, 1]*w2 + noise*np.random.randn(m)
    return data

X_train = generate_3d_data(60)
X_train = X_train - X_train.mean(axis=0, keepdims=True)   # centre: see the notebox

# no activations anywhere: this is Eq. (12.linae)
encoder = keras.models.Sequential([keras.layers.Dense(2, input_shape=[3])])
decoder = keras.models.Sequential([keras.layers.Dense(3, input_shape=[2])])
autoencoder = keras.models.Sequential([encoder, decoder])
autoencoder.compile(loss="mse", optimizer=keras.optimizers.SGD(learning_rate=1.5))
autoencoder.fit(X_train, X_train, epochs=200, verbose=0)   # target == input

codings = encoder.predict(X_train)
# compare with PCA: the subspaces should agree, the bases need not
U, s, Vt = np.linalg.svd(X_train, full_matrices=False)
print("PCA loadings:\n", Vt[:2].T)
print("encoder weights:\n", encoder.layers[0].get_weights()[0])


The final two prints are Proposition prop:12-nonunique in action: the two
matrices will differ, and their column spaces will not.

A stacked, nonlinear autoencoder on MNIST, with a mirrored decoder and a
logistic output to match the $[0,1]$ data:


In [ ]:
(X_train_full, _), (X_test, _) = keras.datasets.mnist.load_data()
X_train_full = X_train_full.astype(np.float32) / 255
X_test = X_test.astype(np.float32) / 255
X_train, X_valid = X_train_full[:-5000], X_train_full[-5000:]

stacked_encoder = keras.models.Sequential([
    keras.layers.Flatten(input_shape=[28, 28]),
    keras.layers.Dense(100, activation="selu"),
    keras.layers.Dense(30, activation="selu"),          # the bottleneck, p = 30
])
stacked_decoder = keras.models.Sequential([             # the exact mirror
    keras.layers.Dense(100, activation="selu", input_shape=[30]),
    keras.layers.Dense(28 * 28, activation="sigmoid"),  # data live in [0,1]
    keras.layers.Reshape([28, 28]),
])
stacked_ae = keras.models.Sequential([stacked_encoder, stacked_decoder])
stacked_ae.compile(loss="binary_crossentropy",          # Eq. (12.bce)
                   optimizer=keras.optimizers.Adam(1e-3))
stacked_ae.fit(X_train, X_train, epochs=10,
               validation_data=(X_valid, X_valid))


The convolutional version reuses Chapter 10 without modification.
The encoder is a stack of strided convolutions, which by
Proposition prop:10-arith halve the spatial size each time; the decoder
mirrors it with transposed convolutions, which are the adjoint operation
discussed in the notebox of Section *Backpropagation through a convolutional layer*.


In [ ]:
# Encoder: (28,28,1) -> (14,14,16) -> (7,7,32) -> (4,4,64), Eq. (10.outsize)
conv_encoder = keras.models.Sequential([
    keras.layers.Reshape([28, 28, 1], input_shape=[28, 28]),
    keras.layers.Conv2D(16, 3, strides=2, padding="SAME", activation="selu"),
    keras.layers.Conv2D(32, 3, strides=2, padding="SAME", activation="selu"),
    keras.layers.Conv2D(64, 3, strides=2, padding="SAME", activation="selu"),
])
# Decoder: the exact mirror, transposed convolutions doubling the size
conv_decoder = keras.models.Sequential([
    keras.layers.Conv2DTranspose(32, 3, strides=2, padding="SAME",
                                 activation="selu", input_shape=[4, 4, 64]),
    keras.layers.Conv2DTranspose(16, 3, strides=2, padding="SAME",
                                 activation="selu"),
    keras.layers.Conv2DTranspose(1, 3, strides=2, padding="SAME",
                                 activation="sigmoid"),
    keras.layers.Lambda(lambda x: x[:, :28, :28, :]),   # crop 32 -> 28
    keras.layers.Reshape([28, 28]),
])
conv_ae = keras.models.Sequential([conv_encoder, conv_decoder])
conv_ae.compile(loss="binary_crossentropy",
                optimizer=keras.optimizers.Adam(1e-3))


A recurrent autoencoder reuses Chapter 11 in the same way, reading
each image as a sequence of $28$ rows.  The encoder's final state is the code;
\verb!RepeatVector! hands it to the decoder once per output step.


In [ ]:
recurrent_encoder = keras.models.Sequential([
    keras.layers.LSTM(100, return_sequences=True, input_shape=[28, 28]),
    keras.layers.LSTM(30),                       # final state = the code
])
recurrent_decoder = keras.models.Sequential([
    keras.layers.RepeatVector(28, input_shape=[30]),
    keras.layers.LSTM(100, return_sequences=True),
    keras.layers.TimeDistributed(keras.layers.Dense(28, activation="sigmoid")),
])
recurrent_ae = keras.models.Sequential([recurrent_encoder, recurrent_decoder])
recurrent_ae.compile(loss="binary_crossentropy",
                     optimizer=keras.optimizers.Adam(1e-3))


The same dense autoencoder in PyTorch, where the mirror symmetry is explicit:


In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST("./data", train=True, download=True,
                   transform=transforms.ToTensor()),
    batch_size=128, shuffle=True)


class Autoencoder(nn.Module):
    """784 -> 256 -> 128 -> 256 -> 784, an exact mirror; Eq. (12.factor)."""
    def __init__(self, x_dim=784, h_dim=256, z_dim=128):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(x_dim, h_dim), nn.BatchNorm1d(h_dim), nn.ReLU(),
            nn.Linear(h_dim, z_dim), nn.BatchNorm1d(z_dim), nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(z_dim, h_dim), nn.BatchNorm1d(h_dim), nn.ReLU(),
            nn.Linear(h_dim, x_dim), nn.Sigmoid(),       # data in [0,1]
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


model = Autoencoder()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

for epoch in range(1, 101):
    model.train()
    total = 0.0
    for data, _ in train_loader:
        inputs = data.view(-1, 784)
        loss = loss_fn(model(inputs), inputs)            # target == input
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        total += loss.item()
    if epoch % 10 == 0:
        print(f"epoch {epoch:3d}  train {total/len(train_loader):.6f}")


Turning this into a denoising autoencoder, Eq. (12.17), is one
line: replace the forward call by
\verb!model(inputs + 0.3*torch.randn_like(inputs))! while leaving the target as
\verb!inputs!.  A sparse autoencoder, Eq. (12.19), adds
\verb!lam * model.encoder(inputs).abs().sum()! to the loss.  Neither requires a
new architecture, which is the point of Section *Regularised autoencoders*.


## Summary and the programs

An autoencoder approximates the identity through a constrained factorisation,
and everything interesting follows from the constraint rather than from the
objective.

The linear case is solved completely.  Lemma lem:12-projector reduces the
rank-constrained problem to orthogonal projectors, Lemma lem:12-trace
turns reconstruction error into variance not captured, and
Theorem thm:12-kyfan maximises that variance at the leading eigenvectors.
Theorem thm:12-aepca assembles these: the minimum error is the eigenvalue
tail $\sum_{i>p}\lambda_i$ and the learned map is the PCA projector.  We
measured all of it -- the error floor reproduced to six digits, the singular
values of $\bm{W}_e\bm{W}_d$ equal to three ones and five zeros to machine
precision, the subspaces agreeing to a fifth of a degree.

Proposition prop:12-nonunique is the caveat to carry away.  The
factorisation is invariant under $\bm{W}_e\to\bm{W}_e\bm{M}$,
$\bm{W}_d\to\bm{M}^{-1}\bm{W}_d$, so the autoencoder identifies the principal
*subspace* and not the principal *directions*.  Measured:
$\|\bm{W}_e-\bm{U}_p\|_F=1.87$, while the distance from $\bm{W}_e$ to the span
of $\bm{U}_p$ is $2.5\times10^{-3}$.  If the directions matter, use PCA.

Nonlinearity buys curvature, at the cost of every guarantee.  On a helix in
$\mathbb{R}^{3}$ a one-dimensional nonlinear code beat a two-dimensional linear
one, $0.201$ against $0.332$, and at equal $p$ the margin was a factor of four
to fifty.  But Figure fig:aenonlinear(c) showed the reconstruction to be
excellent while the code was a poor coordinate, saturating and jumping.
Equation (12.2) scores $\hat{\bm{x}}$ and never mentions $\bm{z}$,
so a good representation is something one must ask for explicitly -- by
denoising, sparsity, contraction, or by moving to a probabilistic model.

The programs are in the directory  

`doc/BookML/BookPrograms/chapter12_autoencoders`.  

Every listing above appears there as a numbered file, and three modules run
start to finish and reproduce the numbers quoted in the text:

- `ae.py` -- the forward and backward passes, Adam, the encode
   helper, and `pca` for comparison.
- `verify_ae.py` -- the gradient checks of
   Section *Verifying the theorem* and the linear-autoencoder-versus-PCA
   comparison, including the principal angles and the singular values of
   $\bm{W}_e\bm{W}_d$.
- `run_nonlinear.py` -- the helix experiment of
   Table 12.1.

The figures are generated by `ch12_figures.py` in
`doc/BookML/BookFigures`; neither is drawn by hand.


## Exercises

### Warm-up exercises

1. **The rank factorisation.**
   Show that every $\bm{A}\in\mathbb{R}^{d\times d}$ with
   $\operatorname{rank}(\bm{A})\le p$ can be written as $\bm{W}_e\bm{W}_d$ with
   $\bm{W}_e\in\mathbb{R}^{d\times p}$ and $\bm{W}_d\in\mathbb{R}^{p\times d}$.
   This is the step that turns Eq. (12.8) into a statement
   about ranks rather than about networks.
2. **Projection is optimal.**
   Prove directly that for a subspace $\mathcal{M}$ and a point $\bm{x}$, the
   minimiser of $\|\bm{x}-\bm{m}\|_2$ over $\bm{m}\in\mathcal{M}$ is the
   orthogonal projection, and that the residual is orthogonal to $\mathcal{M}$.
   Where exactly is this used in Lemma lem:12-projector?
3. **Trace identities.**
   Verify each step of Lemma lem:12-trace, stating which property of the
   trace or of $\bm{P}$ you use at each equality.  Then show that
   $\operatorname{Tr}(\bm{S})=\sum_i\lambda_i$ for symmetric $\bm{S}$.
4. **Ky Fan by hand.**
   For $d=3$, $p=2$ and
   $\bm{\Lambda}=\operatorname{diag}(5,3,1)$, verify Theorem thm:12-kyfan
   by direct computation, and exhibit two different $\bm{B}$ attaining the
   maximum.  What is the set of all maximisers?
5. **The non-uniqueness.**
   Train a linear autoencoder twice from different seeds and compare (a) the
   reconstruction errors, (b) the products $\bm{W}_e\bm{W}_d$, (c) the individual
   matrices $\bm{W}_e$.  Which two agree and which does not, and why?  Then find
   the matrix $\bm{M}$ of Proposition prop:12-nonunique relating the two
   runs and check that $\bm{W}_e^{(1)}\bm{M}\approx\bm{W}_e^{(2)}$.
6. **Overcomplete.**
   Build an autoencoder with $p>d$ and no regularisation, and confirm that it
   drives the reconstruction error to zero while learning nothing.  Exhibit the
   weights that achieve this exactly.  Then add each of the three regularisers of
   Section *Regularised autoencoders* and report what changes.

### Project-style exercise: autoencoders and PCA

**Part a: the machinery.** 
Implement the autoencoder forward and backward passes and verify every gradient
against central differences for at least three architectures, including a purely
linear one.  You will find parameters whose gradient is exactly zero; explain
which and why, and make your checker handle them correctly.

**Part b: the theorem.** 
Reproduce Section *Verifying the theorem* on data of your own construction.  Confirm
Eq. (12.14) to as many digits as your optimiser will give; measure
the principal angles; and confirm that the singular values of
$\bm{W}_e\bm{W}_d$ are $p$ ones and $d-p$ zeros.  Then break the assumption:
make $\lambda_p=\lambda_{p+1}$ exactly and report what happens to part (iii) of
the theorem.

**Part c: how the floor is approached.** 
Study the convergence of the linear autoencoder to $\sum_{i>p}\lambda_i$ as a
function of the learning rate, the initialisation scale, and the eigenvalue gap
$\lambda_p-\lambda_{p+1}$.  Saddle points are known to exist in this landscape
corresponding to subsets of eigenvectors other than the leading ones; can you
find one, and how long does the optimiser linger there?

**Part d: curvature.** 
Reproduce Table 12.1, then vary the curvature of the manifold
continuously -- for instance by interpolating between a straight line and a
helix -- and plot the PCA and autoencoder errors against the curvature.  At what
point does the nonlinear model start to pay for itself?

**Part e: representation, not reconstruction.** 
Reproduce Figure fig:aenonlinear(c) and quantify how badly the code
parameterises the curve, for instance by the Spearman correlation between $z$
and $t$.  Then try to fix it: a wider bottleneck, a different activation, a
contractive penalty, more training.  Report which of these help and which merely
lower the reconstruction error while leaving the code as bad as before.  This is
the central lesson of the chapter and it deserves evidence.

**Part f: the regularisers.** 
Implement the denoising, sparse and contractive objectives of
Eqs. (12.17)--(12.20).  For the contractive
case you will need $\|\bm{J}_f\|_F^{2}$; obtain it by automatic differentiation
and verify it against finite differences.  Compare the three on an overcomplete
architecture where the bottleneck cannot help.

**Part g: convolutional and recurrent.** 
Build a convolutional autoencoder from your Chapter 10 code and a
recurrent one from your Chapter 11 code, on the same data.  Check
the decoder's output sizes against Proposition prop:10-arith before
training.  Compare all three architectures at matched parameter counts and say
which inductive bias the data actually reward.

**Part h: probabilistic.** 
Implement PPCA, Eq. (12.21), by maximum likelihood, and verify
Eq. (12.23) numerically.  Use the likelihood to select $p$ on
data where you know the true latent dimension, and compare against choosing $p$
by the reconstruction-error elbow.  Which is more reliable, and does either
recover the truth?
